# Phase 4: Advanced Causal Inference

Exploring multiple treatments and cost-benefit analysis:
1. Multiple Treatments: TechSupport, Contract Upgrades, and Payment Method Changes.
2. Heterogeneous Treatment Effects (CATE).
3. Cost-Benefit Analysis ($$ saved per intervention).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dowhy import CausalModel
from econml.dr import DRLearner
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
import warnings
warnings.filterwarnings('ignore')


In [2]:
df = pd.read_csv('../data/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

df['T_TechSupport'] = (df['TechSupport'] == 'Yes').astype(int)
df['T_Contract'] = (df['Contract'].isin(['One year', 'Two year'])).astype(int)
df['T_Payment'] = (df['PaymentMethod'].isin(['Bank transfer (automatic)', 'Credit card (automatic)'])).astype(int)


In [3]:
confounders = ['tenure', 'MonthlyCharges', 'SeniorCitizen']
treatments = ['T_TechSupport', 'T_Contract', 'T_Payment']
for t in treatments:
    model = CausalModel(data=df, treatment=t, outcome='Churn', common_causes=confounders)
    identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
    estimate = model.estimate_effect(identified_estimand, method_name='backdoor.propensity_score_matching', target_units='ate')
    print(f'ATE of {t}: {estimate.value:.4f}')


ATE of T_TechSupport: -0.1041
ATE of T_Contract: -0.3332
ATE of T_Payment: -0.1548


In [4]:
X_uplift = df[confounders].values
T_uplift = df['T_Contract'].values
Y_uplift = df['Churn'].values
est = DRLearner(model_regression=GradientBoostingRegressor(), model_propensity=GradientBoostingClassifier(), model_final=GradientBoostingRegressor(), cv=3)
est.fit(Y_uplift, T_uplift, X=X_uplift)
df['Uplift_Contract'] = est.effect(X_uplift)


In [5]:
cost_per_upgrade = 100
revenue_per_retained = 1200
target = df[df['Uplift_Contract'] < -0.1]
num_targets = len(target)
expected_retained = -target['Uplift_Contract'].sum()
total_cost = num_targets * cost_per_upgrade
total_revenue = expected_retained * revenue_per_retained
net_profit = total_revenue - total_cost
roi = (net_profit / total_cost) * 100
print(f'Target Segment Size: {num_targets}')
print(f'Expected Retained: {expected_retained:.1f}')
print(f'Total Cost: ${total_cost:,.2f}')
print(f'Net Profit: ${net_profit:,.2f}')
print(f'ROI: {roi:.2f}%')


Target Segment Size: 4696
Expected Retained: 1417.4
Total Cost: $469,600.00
Net Profit: $1,231,333.95
ROI: 262.21%


In [6]:
df.to_csv('../data/processed_with_advanced_causal.csv', index=False)
